# Train Fusion No2 Model

## About this Notebook

### Objective

Train and evaluate multimodal deep learning models to predict log-transformed NO₂ levels using Sentinel-2 imagery, Sentinel-5P atmospheric data, and ground sensor measurements.

This notebook implements single-modality (S2-only, S5-only) and fusion models to assess the value of combining optical and atmospheric satellite information.

---

### Inputs

- Cleaned ground sensor dataset (from `ExportSensorData`)
- Sentinel-2 image chips (from `ExportS2SensorData`)
- Sentinel-5P NO₂ raster patches (from `ExportS5SensorData`)
- Precomputed normalization percentiles
- Train/validation/test splits

Primary data bucket:
`gs://msads-mba-capstone-team-1/Data/`

Models are defined in:
`fusionmodels.py`

---

### Outputs

- Trained S2-only model
- Trained S5-only model
- Trained Fusion model
- Performance metrics (RMSE, MAE, R²)
- Saved model weights (.pt files) in GCS
- Evaluation plots and diagnostic outputs

Example output path:
`gs://msads-mba-capstone-team-1/Models/`

---

### Workflow Overview

1. Load train/validation/test splits  
2. Initialize datasets and dataloaders  
3. Train S2-only model  
4. Train S5-only model  
5. Train fusion model (late fusion architecture)  
6. Evaluate models on validation and test sets  
7. Save best-performing model weights  

---

### Key Notes

- Target variable is log-transformed NO₂ (`no2_log`).
- All satellite inputs are percentile-normalized to [0, 1].
- Sentinel-2 backbone is a modified ResNet50 with 12 input channels.
- Sentinel-5 branch uses a lightweight CNN feature extractor.
- Fusion model concatenates learned 2048-d feature embeddings from each branch.
- Reproducibility requires fixed random seeds and consistent data splits.

---

### Pipeline Position

ExportSensorData  
→ ExportS2SensorData  
→ ExportS5SensorData  
→ FusionNO2Model  

This notebook produces the core predictive model used for emissions verification and downstream analysis.

## Config

In [1]:
# Input Paths
fusion_model_folder_path = "/home/jupyter/msads-mba-capstone-team-1/Notebooks/team_notebooks/PIPELINE"
s2_path = "msads-mba-capstone-team-1/Data/TrainingData/S2_images"
s5_path = "msads-mba-capstone-team-1/Data/TrainingData/S5_images"
sensor_data_path = "gs://msads-mba-capstone-team-1/Data/TrainingData/sensor_data_clean.csv"
base_path = "gs://msads-mba-capstone-team-1/Data/TrainingData/" # for where data is stored

# Save Paths
## For Percentiles
save_s2_path = "msads-mba-capstone-team-1/Data/TrainingData/s2_percentiles.npy"
save_s5_path = "msads-mba-capstone-team-1/Data/TrainingData/s5_percentiles.npy"
BASE_MODEL_DIR = "msads-mba-capstone-team-1/models"
LOCAL_MODEL_DIR = "local_models"

## Setup

In [2]:
import torch
from torch.utils.data import Dataset
import torch.nn as nn
import torchvision.models as models
import rasterio
import gcsfs
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import random
from tqdm import tqdm
from torch.utils.data import DataLoader
import datetime
import time
import os
import sys
import json
import ipykernel
from notebook import notebookapp
import urllib.request
import torch.optim as optim
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:
import sys

sys.path.insert(0, fusion_model_folder_path)

from fusionmodels import (
    S2Dataset,
    S2Model,
    S5OnlyDataset,
    S5Model,
    S5FeatureExtractor,
    FusionDataset,
    FusionModel
)

# Read-In Data

In [5]:
# Read in Sensor Data
import pandas as pd
sensor_data = pd.read_csv(
    sensor_data_path
)

In [6]:
# Set Paths for S2 and S5 Images around Sensor Data
import gcsfs
fs = gcsfs.GCSFileSystem()
s2_files = [f for f in fs.ls(s2_path) if f.endswith(".tif")]

In [7]:
# Load Train/Test Split If It Exists OR Save
import pandas as pd
from sklearn.model_selection import train_test_split
import gcsfs

# ----------------------------
# GCS setup
# ----------------------------
train_path = base_path + "train_split.csv"
val_path   = base_path + "val_split.csv"
test_path  = base_path + "test_split.csv"

fs = gcsfs.GCSFileSystem()

def gcs_file_exists(path):
    try:
        return fs.exists(path)
    except Exception:
        return False

# ----------------------------
# Load if exists, else create
# ----------------------------
if (
    gcs_file_exists(train_path)
    and gcs_file_exists(val_path)
    and gcs_file_exists(test_path)
):
    print("Existing splits found. Loading from GCS...")
    
    train_df = pd.read_csv(train_path)
    val_df   = pd.read_csv(val_path)
    test_df  = pd.read_csv(test_path)

else:
    print("No existing splits found. Creating new splits...")
    
    train_df, temp_df = train_test_split(
        sensor_data,
        test_size=0.3,
        random_state=42
    )
    
    val_df, test_df = train_test_split(
        temp_df,
        test_size=0.5,
        random_state=42
    )
    
    # Save to GCS
    train_df.to_csv(train_path, index=False)
    val_df.to_csv(val_path, index=False)
    test_df.to_csv(test_path, index=False)
    
    print("Splits created and saved to GCS.")

print(f"Train: {len(train_df)}")
print(f"Val:   {len(val_df)}")
print(f"Test:  {len(test_df)}")

No existing splits found. Creating new splits...
Splits created and saved to GCS.
Train: 475
Val:   102
Test:  102


In [8]:
# Normalize images
# ---------------------------------------------------
# CLEAN TRAIN-ONLY NORMALIZATION
# ---------------------------------------------------

import numpy as np
import gcsfs
import rasterio
import random
from tqdm import tqdm

fs = gcsfs.GCSFileSystem()

# ---------------------------------------------------
# Build TRAIN file lists
# ---------------------------------------------------

train_s2_files = [
    f"{s2_path}/S2_Device{row.DeviceId}_{row.quarter}.tif"
    for _, row in train_df.iterrows()
]

train_s5_files = [
    f"{s5_path}/S5_Device{row.DeviceId}_{row.quarter}.tif"
    for _, row in train_df.iterrows()
]

# Random sampling (important)
sample_size = min(200, len(train_s2_files))
sample_s2_files = random.sample(train_s2_files, sample_size)
sample_s5_files = random.sample(train_s5_files, sample_size)

# ---------------------------------------------------
# Compute S2 Percentiles (12 bands)
# ---------------------------------------------------

print("Computing S2 percentiles from TRAIN data...")

s2_pixels = [[] for _ in range(12)]

for file in tqdm(sample_s2_files):
    with fs.open(file, 'rb') as f:
        with rasterio.open(f) as src:
            data = src.read().astype(np.float32)  # (12,120,120)
    
    for b in range(12):
        s2_pixels[b].append(data[b].flatten())

s2_percentiles = []

for b in range(12):
    band_data = np.concatenate(s2_pixels[b])
    p1 = np.percentile(band_data, 1)
    p99 = np.percentile(band_data, 99)
    s2_percentiles.append((float(p1), float(p99)))

print("S2 percentiles computed.")


# ---------------------------------------------------
# Compute S5 Percentiles (single band)
# ---------------------------------------------------

print("Computing S5 percentiles from TRAIN data...")

s5_pixels = []

for file in tqdm(sample_s5_files):
    with fs.open(file, 'rb') as f:
        with rasterio.open(f) as src:
            data = src.read(1).astype(np.float32)  # (120,120)
    
    s5_pixels.append(data.flatten())

s5_pixels = np.concatenate(s5_pixels)

s5_p1 = float(np.percentile(s5_pixels, 1))
s5_p99 = float(np.percentile(s5_pixels, 99))

print("S5 percentiles computed.")
print("S5 P1:", s5_p1)
print("S5 P99:", s5_p99)


# ---------------------------------------------------
# Save to GCS
# ---------------------------------------------------

with fs.open(save_s2_path, "wb") as f:
    np.save(f, np.array(s2_percentiles))

with fs.open(save_s5_path, "wb") as f:
    np.save(f, np.array([s5_p1, s5_p99]))

print("TRAIN-only S2 and S5 percentiles saved successfully.")

Computing S2 percentiles from TRAIN data...


100%|██████████| 200/200 [00:07<00:00, 26.04it/s]


S2 percentiles computed.
Computing S5 percentiles from TRAIN data...


100%|██████████| 200/200 [00:08<00:00, 22.37it/s]


S5 percentiles computed.
S5 P1: 4.6676344936713576e-05
S5 P99: 0.00010066592949442565
TRAIN-only S2 and S5 percentiles saved successfully.


In [9]:
# Load Percentiles If They Already Exist
import numpy as np
import gcsfs

fs = gcsfs.GCSFileSystem()

with fs.open(save_s2_path, "rb") as f:
    s2_percentiles = np.load(f, allow_pickle=True)

with fs.open(save_s5_path, "rb") as f:
    s5_percentiles = np.load(f, allow_pickle=True)

# S2-Only Model

In [10]:
from torch.utils.data import DataLoader

train_dataset = S2Dataset(train_df, s2_path, s2_percentiles)
val_dataset   = S2Dataset(val_df, s2_path, s2_percentiles)
test_dataset  = S2Dataset(test_df, s2_path, s2_percentiles)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=16)
test_loader  = DataLoader(test_dataset, batch_size=16)

In [11]:
model = S2Model().to(device)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [12]:
import datetime
import time
import gcsfs
import os
from sklearn.metrics import r2_score, mean_absolute_error
import numpy as np

fs = gcsfs.GCSFileSystem()

model_type = "S2"
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

local_model_path = f"best_{model_type}_model_{timestamp}.pt"
gcs_model_path = f"msads-mba-capstone-team-1/models/{local_model_path}"

num_epochs = 100
best_val_loss = float("inf")
patience = 10
patience_counter = 0

print(f"\nStarting training at {timestamp}")
print(f"Model type: {model_type}")
print(f"Device: {device}")
print("="*60)

for epoch in range(num_epochs):
    
    epoch_start_time = time.time()
    
    # --------------------
    # TRAIN
    # --------------------
    model.train()
    train_losses = []
    
    print(f"\nEpoch {epoch+1}/{num_epochs}")
    
    for batch_idx, (s2, labels) in enumerate(train_loader):
        s2 = s2.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        
        preds = model(s2).squeeze()
        loss = criterion(preds, labels)
        
        loss.backward()
        optimizer.step()
        
        train_losses.append(loss.item())
        
        # Print every 10 batches
        if (batch_idx + 1) % 10 == 0:
            print(f"  Batch {batch_idx+1}/{len(train_loader)} | "
                  f"Running Loss: {np.mean(train_losses):.4f}")
    
    train_loss = np.mean(train_losses)
    
    # --------------------
    # VALIDATION
    # --------------------
    model.eval()
    val_losses = []
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for s2, labels in val_loader:
            s2 = s2.to(device)
            labels = labels.to(device)
            
            preds = model(s2).squeeze()
            loss = criterion(preds, labels)
            
            val_losses.append(loss.item())
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    val_loss = np.mean(val_losses)
    r2 = r2_score(all_labels, all_preds)
    mae = mean_absolute_error(all_labels, all_preds)
    
    epoch_time = time.time() - epoch_start_time
    
    print(f"\nEpoch {epoch+1} Complete")
    print(f"  Train Loss: {train_loss:.4f}")
    print(f"  Val Loss:   {val_loss:.4f}")
    print(f"  Val R2:     {r2:.4f}")
    print(f"  Val MAE:    {mae:.4f}")
    print(f"  Epoch Time: {epoch_time:.2f} sec")
    print("-"*60)
    
    # --------------------
    # EARLY STOPPING + SAVE
    # --------------------
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        
        torch.save(model.state_dict(), local_model_path)
        
        with fs.open(gcs_model_path, "wb") as f:
            with open(local_model_path, "rb") as local_f:
                f.write(local_f.read())
        
        print(f"   New best model saved!")
        print(f"   Path: {gcs_model_path}")
        print(f"   Best Val Loss: {best_val_loss:.4f}")
        
    else:
        patience_counter += 1
        print(f"  No improvement. Patience: {patience_counter}/{patience}")
    
    if patience_counter >= patience:
        print("\n  Early stopping triggered.")
        break

print("\n Training complete.")



Starting training at 20260301_211420
Model type: S2
Device: cuda

Epoch 1/100
  Batch 10/30 | Running Loss: 4.0232
  Batch 20/30 | Running Loss: 2.4603
  Batch 30/30 | Running Loss: 1.7031

Epoch 1 Complete
  Train Loss: 1.7031
  Val Loss:   0.2144
  Val R2:     -0.8189
  Val MAE:    0.3595
  Epoch Time: 22.65 sec
------------------------------------------------------------
   New best model saved!
   Path: msads-mba-capstone-team-1/models/best_S2_model_20260301_211420.pt
   Best Val Loss: 0.2144

Epoch 2/100
  Batch 10/30 | Running Loss: 0.2233
  Batch 20/30 | Running Loss: 0.1672
  Batch 30/30 | Running Loss: 0.1521

Epoch 2 Complete
  Train Loss: 0.1521
  Val Loss:   0.1392
  Val R2:     -0.1785
  Val MAE:    0.3131
  Epoch Time: 21.48 sec
------------------------------------------------------------
   New best model saved!
   Path: msads-mba-capstone-team-1/models/best_S2_model_20260301_211420.pt
   Best Val Loss: 0.1392

Epoch 3/100
  Batch 10/30 | Running Loss: 0.1007
  Batch 20

In [15]:
model.load_state_dict(torch.load(f"{BASE_MODEL_DIR}/best_S2_model_20260301_211420.pt"))
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for s2, labels in test_loader:
        s2 = s2.to(device)
        preds = model(s2).squeeze()
        
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

from sklearn.metrics import mean_squared_error

print("Test R2:", r2_score(all_labels, all_preds))
print("Test MSE:", mean_squared_error(all_labels, all_preds))
print("Test MAE:", mean_absolute_error(all_labels, all_preds))

Test R2: 0.5522371560472883
Test MSE: 0.067040317241116
Test MAE: 0.19617794541751638


1. best_S2_model_20260220_050800.pt
- Log NO2, device leakage
- Test R2: 0.3464430780633987
- Test MSE: 0.09785238765006664
- Test MAE: 0.2311727147476346

2. best_S2_model_20260301_211420
- Test R2: 0.5522371560472883
- Test MSE: 0.067040317241116
- Test MAE: 0.19617794541751638

# S5 Model

In [16]:
from torch.utils.data import DataLoader
import pandas as pd
import gcsfs

fs = gcsfs.GCSFileSystem()

train_df = pd.read_csv(train_path)
val_df   = pd.read_csv(val_path)
test_df  = pd.read_csv(test_path)

print(train_df.columns)

train_dataset = S5OnlyDataset(train_df, s5_path, s5_percentiles)
val_dataset   = S5OnlyDataset(val_df, s5_path, s5_percentiles)
test_dataset  = S5OnlyDataset(test_df, s5_path, s5_percentiles)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=16)
test_loader  = DataLoader(test_dataset, batch_size=16)

Index(['DeviceId', 'Latitude', 'Longitude', 'quarter', 'no2_mean', 'pm25_mean',
       'o3_mean', 'num_days', 'q_start', 'q_end', 'no2_log'],
      dtype='object')


In [17]:
import torch
import torch.optim as optim
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import datetime
import os
import time
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = S5Model().to(device)

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

num_epochs = 100
patience = 10
best_val_loss = float("inf")
patience_counter = 0

os.makedirs("local_models", exist_ok=True)

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
local_model_path = f"local_models/best_S5_model_{timestamp}.pt"


In [18]:
import datetime
import time
import gcsfs
import os
from sklearn.metrics import r2_score, mean_absolute_error
import numpy as np

fs = gcsfs.GCSFileSystem()

model_type = "S5"
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

# Local + GCS paths
os.makedirs("local_models", exist_ok=True)

local_model_path = f"local_models/best_{model_type}_model_{timestamp}.pt"
gcs_model_path = f"msads-mba-capstone-team-1/models/best_{model_type}_model_{timestamp}.pt"

num_epochs = 100
best_val_loss = float("inf")
patience = 10
patience_counter = 0

print(f"\n Starting {model_type} training at {timestamp}")
print(f"Device: {device}")
print("="*60)

for epoch in range(num_epochs):
    
    epoch_start_time = time.time()
    
    # --------------------
    # TRAIN
    # --------------------
    model.train()
    train_losses = []
    
    print(f"\nEpoch {epoch+1}/{num_epochs}")
    
    for batch_idx, (s5, labels) in enumerate(train_loader):
        
        s5 = s5.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        
        preds = model(s5).squeeze()
        loss = criterion(preds, labels)
        
        loss.backward()
        optimizer.step()
        
        train_losses.append(loss.item())
        
        if (batch_idx + 1) % 10 == 0:
            print(f"  Batch {batch_idx+1}/{len(train_loader)} | "
                  f"Running Loss: {np.mean(train_losses):.4f}")
    
    train_loss = np.mean(train_losses)
    
    # --------------------
    # VALIDATION
    # --------------------
    model.eval()
    val_losses = []
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for s5, labels in val_loader:
            
            s5 = s5.to(device)
            labels = labels.to(device)
            
            preds = model(s5).squeeze()
            loss = criterion(preds, labels)
            
            val_losses.append(loss.item())
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    val_loss = np.mean(val_losses)
    r2 = r2_score(all_labels, all_preds)
    mae = mean_absolute_error(all_labels, all_preds)
    
    epoch_time = time.time() - epoch_start_time
    
    print(f"\nEpoch {epoch+1} Complete")
    print(f"  Train Loss: {train_loss:.4f}")
    print(f"  Val Loss:   {val_loss:.4f}")
    print(f"  Val R2:     {r2:.4f}")
    print(f"  Val MAE:    {mae:.4f}")
    print(f"  Epoch Time: {epoch_time:.2f} sec")
    print("-"*60)
    
    # --------------------
    # EARLY STOPPING + SAVE
    # --------------------
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        
        # Save locally
        torch.save(model.state_dict(), local_model_path)
        
        # Upload to GCS
        with fs.open(gcs_model_path, "wb") as f:
            with open(local_model_path, "rb") as local_f:
                f.write(local_f.read())
        
        print(f"  New best model saved!")
        print(f"   Local: {local_model_path}")
        print(f"   GCS:   {gcs_model_path}")
        print(f"   Best Val Loss: {best_val_loss:.4f}")
        
    else:
        patience_counter += 1
        print(f"  No improvement. Patience: {patience_counter}/{patience}")
    
    if patience_counter >= patience:
        print("\n  Early stopping triggered.")
        break

print("\n  S5 Training complete.")



 Starting S5 training at 20260301_212835
Device: cuda

Epoch 1/100
  Batch 10/30 | Running Loss: 2.9149
  Batch 20/30 | Running Loss: 1.9792
  Batch 30/30 | Running Loss: 1.4415

Epoch 1 Complete
  Train Loss: 1.4415
  Val Loss:   0.2627
  Val R2:     -1.1544
  Val MAE:    0.4365
  Epoch Time: 23.65 sec
------------------------------------------------------------
  New best model saved!
   Local: local_models/best_S5_model_20260301_212835.pt
   GCS:   msads-mba-capstone-team-1/models/best_S5_model_20260301_212835.pt
   Best Val Loss: 0.2627

Epoch 2/100
  Batch 10/30 | Running Loss: 0.2255
  Batch 20/30 | Running Loss: 0.1947
  Batch 30/30 | Running Loss: 0.1872

Epoch 2 Complete
  Train Loss: 0.1872
  Val Loss:   0.1057
  Val R2:     0.1627
  Val MAE:    0.2546
  Epoch Time: 20.80 sec
------------------------------------------------------------
  New best model saved!
   Local: local_models/best_S5_model_20260301_212835.pt
   GCS:   msads-mba-capstone-team-1/models/best_S5_model_2026

In [20]:
# Load best model
model.load_state_dict(torch.load(f"{BASE_MODEL_DIR}/best_S5_model_20260301_212835.pt"))
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for s5, labels in test_loader:
        s5 = s5.to(device)
        labels = labels.to(device)
        
        preds = model(s5).squeeze()
        
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

test_r2 = r2_score(all_labels, all_preds)
test_mse = mean_squared_error(all_labels, all_preds)
test_mae = mean_absolute_error(all_labels, all_preds)

print("\n===== S5 TEST RESULTS =====")
print(f"Test R2:  {test_r2:.4f}")
print(f"Test MSE: {test_mse:.4f}")
print(f"Test MAE: {test_mae:.4f}")


===== S5 TEST RESULTS =====
Test R2:  0.1674
Test MSE: 0.1247
Test MAE: 0.2649


best_S5_model_20260220_051833.pt
- Test R2:  0.1795
- Test MSE: 0.1229
- Test MAE: 0.2469

best_S5_model_20260301_200400.pt
- Test R2:  0.0304
- Test MSE: 0.1408
- Test MAE: 0.2728

best_S5_model_20260301_212835.pt
===== S5 TEST RESULTS =====
- Test R2:  0.1674
- Test MSE: 0.1247
- Test MAE: 0.2649

# Fusion Model

In [21]:
#1. Config
import os
import datetime

# ==============================
# FUSION CONFIG
# ==============================

CONFIG = {
    "s2_weights": "best_S2_model_20260301_211420.pt",   # filename only
    "s5_weights": "best_S5_model_20260301_212835.pt",   # filename only
    
    "freeze_s2": False,
    "freeze_s5": False,
    
    "learning_rate": 51e-5,   # lower for fine-tuning
    "batch_size": 16,
    "num_epochs": 100,
    "patience": 10
}

os.makedirs(LOCAL_MODEL_DIR, exist_ok=True)

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")


In [22]:
# Load Model From GCS Definition
import io
import gcsfs
import torch

fs = gcsfs.GCSFileSystem()

def load_model_from_gcs(gcs_path, device):
    with fs.open(gcs_path, "rb") as f:
        buffer = io.BytesIO(f.read())
    return torch.load(buffer, map_location=device)

In [23]:
# Load Models

# Load S2
s2_model = S2Model().to(device)
s2_weight_path = f"{BASE_MODEL_DIR}/{CONFIG['s2_weights']}"
s2_model.load_state_dict(load_model_from_gcs(s2_weight_path, device))
s2_model.eval()
s2_backbone = s2_model.backbone


# Load S5
s5_model = S5Model().to(device)
s5_weight_path = f"{BASE_MODEL_DIR}/{CONFIG['s5_weights']}"
s5_model.load_state_dict(load_model_from_gcs(s5_weight_path, device))
s5_model.eval()

s5_backbone = S5FeatureExtractor(s5_model)


In [24]:
# Freeze Logic
if CONFIG["freeze_s2"]:
    for p in s2_backbone.parameters():
        p.requires_grad = False

if CONFIG["freeze_s5"]:
    for p in s5_backbone.parameters():
        p.requires_grad = False

In [25]:
# Fusion Model
model = FusionModel(s2_backbone, s5_backbone).to(device)

In [26]:
# Optimizer
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=CONFIG["learning_rate"]
)

criterion = nn.MSELoss()

In [27]:
from torch.utils.data import DataLoader

train_dataset = FusionDataset(
    train_df,
    s2_path,
    s5_path,
    s2_percentiles,
    s5_percentiles
)

val_dataset = FusionDataset(
    val_df,
    s2_path,
    s5_path,
    s2_percentiles,
    s5_percentiles
)

test_dataset = FusionDataset(
    test_df,
    s2_path,
    s5_path,
    s2_percentiles,
    s5_percentiles
)

train_loader = DataLoader(train_dataset, batch_size=CONFIG["batch_size"], shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=CONFIG["batch_size"])
test_loader  = DataLoader(test_dataset, batch_size=CONFIG["batch_size"])

In [28]:
import time
import numpy as np
import gcsfs
from sklearn.metrics import r2_score, mean_absolute_error

fs = gcsfs.GCSFileSystem()

local_model_path = f"{LOCAL_MODEL_DIR}/best_FUSION_model_{timestamp}.pt"
gcs_model_path = f"{BASE_MODEL_DIR}/best_FUSION_model_{timestamp}.pt"

best_val_loss = float("inf")
patience_counter = 0

print("\n==============================")
print("Starting Fusion Training")
print("CONFIG:", CONFIG)
print("==============================")

for epoch in range(CONFIG["num_epochs"]):
    
    epoch_start = time.time()
    
    # ---------------------
    # TRAIN
    # ---------------------
    model.train()
    train_losses = []
    
    for batch_idx, (s2, s5, labels) in enumerate(train_loader):
        
        s2 = s2.to(device)
        s5 = s5.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        
        preds = model(s2, s5).squeeze()
        loss = criterion(preds, labels)
        
        loss.backward()
        optimizer.step()
        
        train_losses.append(loss.item())
        
        if (batch_idx + 1) % 10 == 0:
            print(f"  Batch {batch_idx+1}/{len(train_loader)} | "
                  f"Running Train Loss: {np.mean(train_losses):.4f}")
    
    train_loss = np.mean(train_losses)
    
    # ---------------------
    # VALIDATION
    # ---------------------
    model.eval()
    val_losses = []
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for s2, s5, labels in val_loader:
            
            s2 = s2.to(device)
            s5 = s5.to(device)
            labels = labels.to(device)
            
            preds = model(s2, s5).squeeze()
            loss = criterion(preds, labels)
            
            val_losses.append(loss.item())
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    val_loss = np.mean(val_losses)
    val_r2 = r2_score(all_labels, all_preds)
    val_mae = mean_absolute_error(all_labels, all_preds)
    
    epoch_time = time.time() - epoch_start
    
    print(f"\nEpoch {epoch+1}/{CONFIG['num_epochs']}")
    print(f"  Train Loss: {train_loss:.4f}")
    print(f"  Val Loss:   {val_loss:.4f}")
    print(f"  Val R2:     {val_r2:.4f}")
    print(f"  Val MAE:    {val_mae:.4f}")
    print(f"  Time:       {epoch_time:.2f}s")
    print("-"*60)
    
    # ---------------------
    # EARLY STOPPING
    # ---------------------
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        
        torch.save(model.state_dict(), local_model_path)
        
        with fs.open(gcs_model_path, "wb") as f:
            with open(local_model_path, "rb") as local_f:
                f.write(local_f.read())
        
        print("    New best Fusion model saved!")
        print("     Path:", gcs_model_path)
        
    else:
        patience_counter += 1
        print(f"  No improvement. Patience: {patience_counter}/{CONFIG['patience']}")
    
    if patience_counter >= CONFIG["patience"]:
        print("\n  Early stopping triggered.")
        break

print("\nFusion Training Complete.")


Starting Fusion Training
CONFIG: {'s2_weights': 'best_S2_model_20260301_211420.pt', 's5_weights': 'best_S5_model_20260301_212835.pt', 'freeze_s2': False, 'freeze_s5': False, 'learning_rate': 0.00051, 'batch_size': 16, 'num_epochs': 100, 'patience': 10}
  Batch 10/30 | Running Train Loss: 1.0125
  Batch 20/30 | Running Train Loss: 0.5783
  Batch 30/30 | Running Train Loss: 0.4259

Epoch 1/100
  Train Loss: 0.4259
  Val Loss:   0.1704
  Val R2:     -0.4065
  Val MAE:    0.3557
  Time:       40.24s
------------------------------------------------------------
    New best Fusion model saved!
     Path: msads-mba-capstone-team-1/models/best_FUSION_model_20260301_215958.pt
  Batch 10/30 | Running Train Loss: 0.1118
  Batch 20/30 | Running Train Loss: 0.0922
  Batch 30/30 | Running Train Loss: 0.0913

Epoch 2/100
  Train Loss: 0.0913
  Val Loss:   0.0788
  Val R2:     0.3946
  Val MAE:    0.2094
  Time:       37.42s
------------------------------------------------------------
    New best Fu

In [31]:
model.load_state_dict(torch.load(f"{BASE_MODEL_DIR}/best_FUSION_model_20260301_052916.pt"))
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for s2, s5, labels in test_loader:
        s2 = s2.to(device)
        s5 = s5.to(device)
        labels = labels.to(device)
        
        preds = model(s2, s5).squeeze()
        
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

test_r2 = r2_score(all_labels, all_preds)
test_mse = np.mean((np.array(all_preds) - np.array(all_labels))**2)
test_mae = mean_absolute_error(all_labels, all_preds)

print("\n===== FUSION TEST RESULTS =====")
print(f"Test R2:  {test_r2:.4f}")
print(f"Test MSE: {test_mse:.4f}")
print(f"Test MAE: {test_mae:.4f}")


===== FUSION TEST RESULTS =====
Test R2:  0.5562
Test MSE: 0.0664
Test MAE: 0.1961


1. 
best_FUSION_model_20260301_215958
- Test R2:  0.7592
- Test MSE: 0.0361
- Test MAE: 0.1368

## Results

1.best_FUSION_model_20260220_054518.pt

CONFIG = {
    "s2_weights": "best_S2_model_20260220_050800.pt",   # filename only
    "s5_weights": "best_S5_model_20260220_051833.pt",   # filename only
    
    "freeze_s2": False,
    "freeze_s5": False,
    
    "learning_rate": 1e-5,   # lower for fine-tuning
    "batch_size": 16,
    "num_epochs": 100,
    "patience": 10
}

===== FUSION TEST RESULTS =====
- Test R2:  0.5546
- Test MSE: 0.0667
- Test MAE: 0.1873

2. best_FUSION_model_20260301_000803.pt
CONFIG = {
    "s2_weights": "best_S2_model_20260220_050800.pt",   # filename only
    "s5_weights": "best_S5_model_20260220_051833.pt",   # filename only
    
    "freeze_s2": True,
    "freeze_s5": False,
    
    "learning_rate": 1e-5,   # lower for fine-tuning
    "batch_size": 16,
    "num_epochs": 100,
    "patience": 10
}

===== FUSION TEST RESULTS =====
- Test R2:  0.5912
- Test MSE: 0.0612
- Test MAE: 0.1872

3. best_FUSION_model_20260301_011657.pt
CONFIG = {
    "s2_weights": "best_S2_model_20260220_050800.pt",
    "s5_weights": "best_S5_model_20260220_051833.pt",

    "freeze_s2": False,
    "freeze_s5": True,

    "learning_rate": 1e-5,
    "batch_size": 16,
    "num_epochs": 100,
    "patience": 10
}
===== FUSION TEST RESULTS =====
Test R2:  0.5633
Test MSE: 0.0654
Test MAE: 0.1893

4. best_FUSION_model_20260301_015710.pt
CONFIG = {
    "s2_weights": "best_S2_model_20260220_050800.pt",
    "s5_weights": "best_S5_model_20260220_051833.pt",

    "freeze_s2": True,
    "freeze_s5": True,

    "learning_rate": 1e-5,
    "batch_size": 16,
    "num_epochs": 100,
    "patience": 10
}
===== FUSION TEST RESULTS =====
- Test R2:  0.5544
- Test MSE: 0.0667
- Test MAE: 0.1920

5. best_FUSION_model_20260301_015710.pt
CONFIG = {
    "s2_weights": "best_S2_model_20260220_050800.pt",   # filename only
    "s5_weights": "best_S5_model_20260220_051833.pt",   # filename only
    
    "freeze_s2": True,
    "freeze_s5": False,
    
    "learning_rate": 1e-6,   # lower for fine-tuning
    "batch_size": 16,
    "num_epochs": 100,
    "patience": 10
}
===== FUSION TEST RESULTS =====
- Test R2:  0.4893
- Test MSE: 0.0765
- Test MAE: 0.2142

6. best_FUSION_model_20260301_052916.pt
CONFIG = {
    "s2_weights": "best_S2_model_20260220_050800.pt",   # filename only
    "s5_weights": "best_S5_model_20260220_051833.pt",   # filename only
    
    "freeze_s2": True,
    "freeze_s5": False,
    
    "learning_rate": 5e-5,   # lower for fine-tuning
    "batch_size": 16,
    "num_epochs": 100,
    "patience": 10
}
===== FUSION TEST RESULTS =====
- Test R2:  0.5930
- Test MSE: 0.0609
- Test MAE: 0.1725

### best model: best_FUSION_model_20260301_052916.pt 
config:

CONFIG = {

    "s2_weights": "best_S2_model_20260220_050800.pt",
    "s5_weights": "best_S5_model_20260220_051833.pt",
    
    "freeze_s2": True,
    "freeze_s5": False,
    
    "learning_rate": 5e-5,   # lower for fine-tuning
    "batch_size": 16,
    "num_epochs": 100,
    "patience": 10
}